# Module 2: When Perfect Training = Disaster

In Module 1 we explored the dataset — 254 countries described by 282 features. Now we ask: **can we just throw all features into a linear regression and predict GDP?**

Spoiler: the model will *memorize* the training data perfectly, yet fail catastrophically on unseen countries. This module builds the intuition for **why**.

In [ ]:
#@title 🔧 Setup { display-mode: "form" }

import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from helpers import load_dataset, ROLE_COLORS, apply_dark_theme, takeaway_box, metric_card, role_bar_colors
from helpers.styling import role_legend_html, styled_button, DARK_BG, AXES_BG, TEXT_COLOR, GREEN, RED, GRAY, MUTED_TEXT

X, y, codebook, role_map = load_dataset()

---
## 1. OLS on All Features

Ordinary Least Squares (OLS) finds the coefficients that minimize the sum of squared errors on the **training** data. With 282 features and only ~200 training samples, the system is *under-determined* — there are infinitely many perfect solutions.

Use the sliders below to pick a test size and random seed, then hit **Train OLS** to see what happens.

In [ ]:
#@title 🔧 OLS Demo Widget { display-mode: "form" }

class OLSDemoWidget:
    """Interactive widget demonstrating OLS overfitting on all features."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.test_slider = widgets.IntSlider(
            value=20, min=10, max=50, step=5,
            description='Test size %:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px')
        )
        self.seed_slider = widgets.IntSlider(
            value=42, min=0, max=100, step=1,
            description='Random seed:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px')
        )
        self.train_btn = styled_button('Train OLS', width='160px')
        self.train_btn.on_click(self._on_train)

        self.output = widgets.Output()

    def _on_train(self, _):
        with self.output:
            clear_output(wait=True)

            test_frac = self.test_slider.value / 100
            seed = self.seed_slider.value

            X_train, X_test, y_train, y_test = train_test_split(
                self.X, self.y, test_size=test_frac, random_state=seed
            )

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model = LinearRegression()
            model.fit(X_train_s, y_train)

            train_r2 = r2_score(y_train, model.predict(X_train_s))
            test_r2 = r2_score(y_test, model.predict(X_test_s))

            train_color = GREEN if train_r2 > 0.8 else RED
            test_color = GREEN if test_r2 > 0.5 else RED

            cards = widgets.HBox([
                metric_card('Train R²', f'{train_r2:.4f}', color=train_color, width='220px'),
                metric_card('Test R²', f'{test_r2:.4f}', color=test_color, width='220px'),
            ])

            info = widgets.HTML(f"""
            <div style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 10px;'>
                Training samples: {len(X_train)} &nbsp;|&nbsp; Test samples: {len(X_test)}
                &nbsp;|&nbsp; Features: {self.X.shape[1]}
            </div>
            """)

            display(cards, info)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>OLS with All 282 Features</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Adjust sliders and click <b>Train OLS</b> to see the train/test gap.
        </p>
        """)
        controls = widgets.VBox([self.test_slider, self.seed_slider, self.train_btn])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch OLS Demo { display-mode: "form" }

ols_demo = OLSDemoWidget(X, y)
display(ols_demo.create_ui())

---
## 2. The Feature Count Experiment

**Key question**: does the problem get worse as we add more features?

When the number of features *p* exceeds the number of training samples *n*, OLS can always find coefficients that achieve a perfect fit on training data — it simply solves an under-determined system. But those coefficients are wildly unstable and produce garbage predictions.

Use the slider to train a single model at a given feature count, or click **Run Sweep** to see the full curve.

In [ ]:
#@title 🔧 Feature Count Widget { display-mode: "form" }

class FeatureCountWidget:
    """Interactive widget showing how test R² degrades as feature count grows."""

    SWEEP_COUNTS = [5, 10, 20, 30, 50, 75, 100, 125, 150, 175, 200, 225, 250, 260, 270, 282]

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.feat_slider = widgets.IntSlider(
            value=50, min=5, max=282, step=5,
            description='# Features:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='400px')
        )
        self.sweep_btn = styled_button('Run Sweep', width='160px')
        self.sweep_btn.on_click(self._on_sweep)

        self.output = widgets.Output()

    def _train_ols(self, n_features):
        """Train OLS with first n_features columns and return train/test R²."""
        X_sub = self.X.iloc[:, :n_features]
        X_train, X_test, y_train, y_test = train_test_split(
            X_sub, self.y, test_size=0.2, random_state=42
        )
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        model = LinearRegression()
        model.fit(X_train_s, y_train)

        train_r2 = r2_score(y_train, model.predict(X_train_s))
        test_r2 = r2_score(y_test, model.predict(X_test_s))
        return train_r2, test_r2

    def _on_sweep(self, _):
        with self.output:
            clear_output(wait=True)

            # Single-value demo from slider
            n_feat = self.feat_slider.value
            single_train_r2, single_test_r2 = self._train_ols(n_feat)

            train_color = GREEN if single_train_r2 > 0.8 else RED
            test_color = GREEN if single_test_r2 > 0.5 else RED

            cards = widgets.HBox([
                metric_card(f'Train R² ({n_feat} feat)', f'{single_train_r2:.4f}',
                            color=train_color, width='220px'),
                metric_card(f'Test R² ({n_feat} feat)', f'{single_test_r2:.4f}',
                            color=test_color, width='220px'),
            ])
            display(cards)

            # Full sweep
            train_scores, test_scores = [], []
            for n in self.SWEEP_COUNTS:
                tr, te = self._train_ols(n)
                train_scores.append(tr)
                test_scores.append(te)

            n_train = int(len(self.y) * 0.8)

            fig, ax = plt.subplots(figsize=(9, 5))
            apply_dark_theme(fig, ax)

            ax.plot(self.SWEEP_COUNTS, train_scores, 'o-', color=GREEN,
                    linewidth=2, markersize=5, label='Train R²')
            ax.plot(self.SWEEP_COUNTS, test_scores, 's-', color=RED,
                    linewidth=2, markersize=5, label='Test R²')

            ax.axvline(x=n_train, color=MUTED_TEXT, linestyle='--', linewidth=1.5, alpha=0.8)
            ax.text(n_train + 3, ax.get_ylim()[0] + 0.1, f'p = n ({n_train})',
                    color=MUTED_TEXT, fontsize=11, va='bottom')

            ax.axhline(y=0, color=GRAY, linestyle=':', linewidth=1, alpha=0.5)

            ax.set_xlabel('Number of Features (p)', fontsize=12)
            ax.set_ylabel('R² Score', fontsize=12)
            ax.set_title('The Hockey Stick of Doom: Train vs Test R²', fontsize=14)
            ax.legend(loc='lower left', fontsize=11, facecolor=AXES_BG, edgecolor='#4a4a6a',
                      labelcolor=TEXT_COLOR)

            plt.tight_layout()
            plt.show()

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Feature Count vs Performance</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Pick a feature count to see its metric cards, then <b>Run Sweep</b> for the full picture.
        </p>
        """)
        controls = widgets.VBox([self.feat_slider, self.sweep_btn])
        return widgets.VBox([title, controls, self.output])

In [ ]:
#@title ▶️ Launch Feature Count Experiment { display-mode: "form" }

feat_widget = FeatureCountWidget(X, y)
display(feat_widget.create_ui())

---
## 3. Actual vs Predicted: A Visual Reality Check

Numbers only tell part of the story. Let us look at the **actual vs predicted** scatter plots side by side for the training and test sets. If the model truly learned the relationship, points should hug the diagonal (y = x) in **both** plots.

In [ ]:
#@title 🔧 Actual vs Predicted Widget { display-mode: "form" }

class ActualVsPredictedWidget:
    """Side-by-side scatter plots of actual vs predicted for train and test."""

    def __init__(self, X, y):
        self.X = X
        self.y = y

        self.show_btn = styled_button('Show Predictions', width='180px')
        self.show_btn.on_click(self._on_show)

        self.output = widgets.Output()

    def _on_show(self, _):
        with self.output:
            clear_output(wait=True)

            X_train, X_test, y_train, y_test = train_test_split(
                self.X, self.y, test_size=0.2, random_state=42
            )

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)

            model = LinearRegression()
            model.fit(X_train_s, y_train)

            y_train_pred = model.predict(X_train_s)
            y_test_pred = model.predict(X_test_s)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            apply_dark_theme(fig, [ax1, ax2])

            # Shared axis limits
            all_vals = np.concatenate([y_train.values, y_test.values,
                                       y_train_pred, y_test_pred])
            vmin, vmax = np.min(all_vals), np.max(all_vals)
            pad = (vmax - vmin) * 0.05

            # Training plot
            ax1.scatter(y_train, y_train_pred, c=GREEN, alpha=0.6, s=30, edgecolors='none')
            ax1.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad],
                     '--', color='white', linewidth=1.5, alpha=0.7)
            ax1.set_xlim(vmin - pad, vmax + pad)
            ax1.set_ylim(vmin - pad, vmax + pad)
            ax1.set_xlabel('Actual GDP per capita (USD)', fontsize=11)
            ax1.set_ylabel('Predicted GDP per capita (USD)', fontsize=11)
            ax1.set_title('Training Set (Perfect Fit)', fontsize=13)

            # Test plot
            ax2.scatter(y_test, y_test_pred, c=RED, alpha=0.6, s=30, edgecolors='none')
            ax2.plot([vmin - pad, vmax + pad], [vmin - pad, vmax + pad],
                     '--', color='white', linewidth=1.5, alpha=0.7)
            ax2.set_xlim(vmin - pad, vmax + pad)
            # Let y-axis auto-scale on test to show how wild predictions can be
            ax2.set_xlabel('Actual GDP per capita (USD)', fontsize=11)
            ax2.set_ylabel('Predicted GDP per capita (USD)', fontsize=11)
            ax2.set_title('Test Set (Wild Predictions)', fontsize=13)

            plt.tight_layout()
            plt.show()

            # Show R² cards too
            train_r2 = r2_score(y_train, y_train_pred)
            test_r2 = r2_score(y_test, y_test_pred)
            cards = widgets.HBox([
                metric_card('Train R²', f'{train_r2:.4f}', color=GREEN, width='220px'),
                metric_card('Test R²', f'{test_r2:.4f}', color=RED, width='220px'),
            ])
            display(cards)

    def create_ui(self):
        title = widgets.HTML(f"""
        <h3 style='color: {TEXT_COLOR}; margin-bottom: 5px;'>Actual vs Predicted: Train &amp; Test</h3>
        <p style='color: {MUTED_TEXT}; font-size: 13px; margin-top: 0;'>
            Click <b>Show Predictions</b> to see scatter plots using all 282 features.
        </p>
        """)
        return widgets.VBox([title, self.show_btn, self.output])

In [ ]:
#@title ▶️ Launch Actual vs Predicted { display-mode: "form" }

avp_widget = ActualVsPredictedWidget(X, y)
display(avp_widget.create_ui())

---
## Takeaway

In [ ]:
#@title 🔧 Takeaway { display-mode: "form" }

display(takeaway_box(
    "With 282 features and 254 countries, OLS memorizes training data perfectly "
    "(R²=1.0) but predicts terribly. <b>Memorization ≠ learning.</b> Can we fix this?"
))